In [ ]:
#===============================================================
# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API
#===============================================================

In [5]:
# =========================
# End-to-End ML Pipeline
# Customer Churn Prediction
# =========================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# -------------------------
# 1. Load Dataset
# -------------------------
df = pd.read_csv("Telco-Customer-Churn.csv")

# -------------------------
# 2. Data Cleaning
# -------------------------
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# -------------------------
# 3. Remove Weak Features
# -------------------------
df.drop(columns=["PhoneService", "MultipleLines"], inplace=True)

# -------------------------
# 4. Features / Target
# -------------------------
X = df.drop("Churn", axis=1)
y = df["Churn"]

# -------------------------
# 5. Train Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 6. Feature Types
# -------------------------
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

# -------------------------
# 7. Preprocessing
# -------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# -------------------------
# 8. Models
# -------------------------
models = {
    "logistic": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    "random_forest": RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    )
}

# -------------------------
# 9. Hyperparameter Grids
# -------------------------
param_grids = {
    "logistic": {
        "model__C": [0.001, 0.01, 0.1, 1, 10],
        "model__solver": ["liblinear", "lbfgs"]
    },

    "random_forest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    }
}

best_models = {}

# -------------------------
# 10. Grid Search Training
# -------------------------
for model_name in models:
    print(f"\nTraining {model_name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", models[model_name])
    ])

    grid_search = GridSearchCV(
        pipeline,
        param_grid=param_grids[model_name],
        cv=5,
        scoring="f1",
        n_jobs=-1,
        verbose=2
    )

    grid_search.fit(X_train, y_train)

    best_models[model_name] = grid_search.best_estimator_

    print("Best Params:", grid_search.best_params_)
    print("Best CV Score:", grid_search.best_score_)

# -------------------------
# 11. Model Evaluation
# -------------------------
best_score = 0
final_model = None

for name, model in best_models.items():
    y_pred = model.predict(X_test)

    print(f"\nEvaluating {name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    f1 = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )["1"]["f1-score"]

    if f1 > best_score:
        best_score = f1
        final_model = model

# -------------------------
# 12. Threshold Tuning
# -------------------------
probs = final_model.predict_proba(X_test)[:, 1]

best_thresh = 0.5
best_f1 = 0

for t in np.arange(0.2, 0.8, 0.01):
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("\nBest Threshold:", best_thresh)
print("Best Threshold F1:", best_f1)

# Final predictions with tuned threshold
final_preds = (probs >= best_thresh).astype(int)

print("\n=== Final Threshold Tuned Results ===")
print("Accuracy:", accuracy_score(y_test, final_preds))
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds))

# -------------------------
# 13. Save Model + Threshold
# -------------------------
joblib.dump(
    {
        "model": final_model,
        "threshold": best_thresh
    },
    "customer_churn_pipeline.pkl"
)

print("\nModel and threshold saved successfully!")


Training logistic...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Params: {'model__C': 10, 'model__solver': 'lbfgs'}
Best CV Score: 0.6277963589926571

Training random_forest...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 10, 'model__n_estimators': 200}
Best CV Score: 0.6366876322260429

Evaluating logistic
Accuracy: 0.7388218594748048
[[743 292]
 [ 76 298]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.80      1035
           1       0.51      0.80      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.75      1409


Evaluating random_forest
Accuracy: 0.7622427253371186
[[809 226]
 [109 265]]
              precision    recall  f1-score   support

           0       0.88      0.78      0.8

In [ ]:
# we use this unseen data values from dataset to check model predict or not .... we see that this will correctly predict
# no or yes (if patterns were clear) which was in dataset but in some cases it not correctly predict yes or no based on customer mixed behaviour just like
# if we enter first row of dataset which output has yes but our model predict no due to mixed behaviour and tp,tn show clearly that. we show two examples one is correct predict and one is wrong


In [8]:
# =========================
# Predict New Customer Churn
# =========================

import pandas as pd
import joblib

# -------------------------
# 1. Load saved model
# -------------------------
saved = joblib.load("customer_churn_pipeline.pkl")

model = saved["model"]
threshold = saved["threshold"]

# ------------------------------------------------------------
# 2. New unseen customer (this correctly predict(0526-SXDJP))
# ------------------------------------------------------------
new_customer = pd.DataFrame([{
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure":72,
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Two year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 42.1,
    "TotalCharges": 2962
}])

# IMPORTANT:
# PhoneService and MultipleLines were removed in training,
# so do not include them here.

# -------------------------
# 3. Predict probability
# -------------------------
prob = model.predict_proba(new_customer)[:, 1][0]

print("Churn Probability:", prob)

# -------------------------
# 4. Apply saved threshold
# -------------------------
prediction = 1 if prob >= threshold else 0

label = "Yes" if prediction == 1 else "No"

print("Prediction:", label)

Churn Probability: 0.018249961741056177
Prediction: No


In [9]:
# =========================
# Predict New Customer Churn
# =========================

import pandas as pd
import joblib

# -------------------------
# 1. Load saved model
# -------------------------
saved = joblib.load("customer_churn_pipeline.pkl")

model = saved["model"]
threshold = saved["threshold"]

# -----------------------------------------------------------------------------------------------------------------------
# 2. New unseen customer (this wrong predict ,output is no but model predict yes due to mixed behaviour )
#    it has low services , tenure 1 , contract, based on this model predict corect that it leave but dataset say no leave
# -----------------------------------------------------------------------------------------------------------------------
new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure":1,
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 29.85,
    "TotalCharges": 29.85
}])

# IMPORTANT:
# PhoneService and MultipleLines were removed in training,
# so do not include them here.

# -------------------------
# 3. Predict probability
# -------------------------
prob = model.predict_proba(new_customer)[:, 1][0]

print("Churn Probability:", prob)

# -------------------------
# 4. Apply saved threshold
# -------------------------
prediction = 1 if prob >= threshold else 0

label = "Yes" if prediction == 1 else "No"

print("Prediction:", label)

Churn Probability: 0.7892627968378103
Prediction: Yes
